In [1]:
import sys
from pathlib import Path

# Add notebooks root to path for utils import
sys.path.insert(0, str(Path.cwd().parent))

from utils import compute_summary_stats, load_aggregate_results

ANALYSIS_SETUP = "default-opus-4-5"

all_results = load_aggregate_results(category="test-generation")
model_df = all_results[ANALYSIS_SETUP]
model_stats = compute_summary_stats(model_df)

print(f"Analysis: {ANALYSIS_SETUP}")
print(f"{'=' * 60}")
print(f"Runs: {model_stats['n_runs']} | Instances: {model_stats['n_instances']}")
print(f"Average execution time: {model_stats['avg_execution_time']:.1f}s ({model_stats['avg_execution_time'] / 60:.1f}m)")
print(f"Mean resolved: {model_stats['mean_resolved']:.2f}%")

model_df.head(n=1)

Analysis: default-opus-4-5
Runs: 5 | Instances: 101
Average execution time: 169.0s (2.8m)
Mean resolved: 45.54%


,run_id,instance_id,resolved,execution_time
0,21494125521,microsoftInternal__NAV-204450,True,168.6


In [2]:
import pandas as pd

from bcbench.config import get_config
from bcbench.dataset import BugFixEntry

_config = get_config()
bcbench_dataset: list[BugFixEntry] = BugFixEntry.load(_config.paths.dataset_dir / "bcbench.jsonl")

dataset_df = pd.DataFrame(
    [
        {
            "instance_id": entry.instance_id,
            "image_count": entry.metadata.image_count or 0,
            "area": entry.metadata.area or "Unknown",
            "project": entry.extract_project_name() or "Unknown",
            "gold_patch": entry.patch,
            "test_patch": entry.test_patch,
        }
        for entry in bcbench_dataset
    ]
)

merged_df = model_df.merge(dataset_df, on="instance_id", how="left")
print(f"Merged {len(merged_df)} results with dataset metadata")
merged_df.head(n=1)

Merged 505 results with dataset metadata


,run_id,instance_id,resolved,execution_time,image_count,area,project,gold_patch,test_patch
0,21494125521,microsoftInternal__NAV-204450,True,168.6,9,finance,BaseApp,diff --git a/App/Layers/W1/BaseApp/Finance/VAT...,diff --git a/App/Layers/W1/Tests/VAT/NonDeduct...


In [3]:
from unidiff import PatchSet


def count_files_in_patch(patch: str) -> int:
    return len(PatchSet(patch))


merged_df["expected_files"] = merged_df["gold_patch"].apply(count_files_in_patch)

bins = [0, 1, float("inf")]
labels = ["1", "2+"]
merged_df["files_bin"] = pd.cut(merged_df["expected_files"], bins=bins, labels=labels)

instance_files_df = (
    merged_df.groupby("instance_id")
    .agg(
        files_bin=("files_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

files_resolution_df = (
    instance_files_df.groupby("files_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

total_instances = files_resolution_df["count"].sum()
files_resolution_df["% Resolved"] = (files_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
files_resolution_df["Instances"] = files_resolution_df["count"].astype(str)
files_resolution_df = files_resolution_df.rename(columns={"files_bin": "Files Changed"})

print('Average "% Resolved" by Number of Files Changed (gold patch):')
print(files_resolution_df[["Files Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Number of Files Changed (gold patch):
Files Changed % Resolved Instances
            1      47.3%        82
           2+      37.9%        19


In [4]:
def count_loc_in_patch(patch: str) -> int:
    patch_set = PatchSet(patch)
    return patch_set.added + patch_set.removed


merged_df["expected_loc"] = merged_df["gold_patch"].apply(count_loc_in_patch)

bins = [0, 5, 10, 50, float("inf")]
labels = ["1-5", "6-10", "11-50", "50+"]
merged_df["loc_bin"] = pd.cut(merged_df["expected_loc"], bins=bins, labels=labels)

instance_loc_df = (
    merged_df.groupby("instance_id")
    .agg(
        loc_bin=("loc_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

loc_resolution_df = (
    instance_loc_df.groupby("loc_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

loc_resolution_df["% Resolved"] = (loc_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
loc_resolution_df["Instances"] = loc_resolution_df["count"].astype(str)
loc_resolution_df = loc_resolution_df.rename(columns={"loc_bin": "LoC Changed"})

print('Average "% Resolved" by Lines of Code Changed (gold patch):')
print(loc_resolution_df[["LoC Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Lines of Code Changed (gold patch):
LoC Changed % Resolved Instances
        1-5      50.9%        44
       6-10      34.0%        10
      11-50      44.4%        36
        50+      38.2%        11


In [5]:
merged_df["project_group"] = merged_df["project"].apply(lambda x: "BaseApp" if x == "BaseApp" else "First-party Apps")

instance_project_df = (
    merged_df.groupby("instance_id")
    .agg(
        project_group=("project_group", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

project_resolution_df = (
    instance_project_df.groupby("project_group", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

project_resolution_df["% Resolved"] = (project_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
project_resolution_df["Instances"] = project_resolution_df["count"].astype(str)
project_resolution_df = project_resolution_df.rename(columns={"project_group": "Project Group"})

print('Average "% Resolved" by Project Group:')
print(project_resolution_df[["Project Group", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Project Group:
   Project Group % Resolved Instances
         BaseApp      44.0%        85
First-party Apps      53.8%        16


In [6]:
bins = [-1, 0, 5, 10, float("inf")]
labels = ["0", "1-5", "6-10", "10+"]
merged_df["image_bin"] = pd.cut(merged_df["image_count"], bins=bins, labels=labels)

instance_image_df = (
    merged_df.groupby("instance_id")
    .agg(
        image_bin=("image_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

image_resolution_df = (
    instance_image_df.groupby("image_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

image_resolution_df["% Resolved"] = (image_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
image_resolution_df["Instances"] = image_resolution_df["count"].astype(str)
image_resolution_df = image_resolution_df.rename(columns={"image_bin": "Image Count"})

print('Average "% Resolved" by Image Count:')
print(image_resolution_df[["Image Count", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Image Count:
Image Count % Resolved Instances
          0      46.5%        34
        1-5      34.7%        30
       6-10      56.3%        27
        10+      46.0%        10
